# 📋 NOTEBOOK #1 — LẬP DANH SÁCH MẪU (Dự án CSRDI)

**Bạn không cần biết code.** Chỉ làm đúng 3 việc:
1. Bấm menu **Runtime → Run all** (Chạy tất cả).
2. Khi hiện bảng xin quyền Google Drive → bấm **Cho phép / Allow**.
3. Cuối cùng file Excel `DanhSach_Mau_CSRDI.xlsx` sẽ tự lưu vào Google Drive của bạn (thư mục `CSRDI`) và tự tải về máy.

Nếu có ô nào báo lỗi màu đỏ → **copy nguyên đoạn đỏ đó gửi lại cho Claude**, đừng lo, đó là chuyện bình thường khi chạy lần đầu.

---

## ⚙️ Ô 1 — Cài thư viện (chạy 1 lần, ~1 phút)

In [ ]:
# === Ô 1 (bản sửa): Cài thư viện + nhập API key vnstock miễn phí ===
!pip install vnstock==4.0.5 openpyxl --quiet

try:
    import vnai
except ImportError:
    from vnstock import vnai

vnai.setup_api_key("vnstock_c38b8853aa3d0d8985b72506865615ef")   # ← xóa chữ này, dán key vào giữa 2 dấu ngoặc kép
vnai.check_api_key_status()

print("✅ Xong. Giờ vào menu 'Thời gian chạy' → 'Chạy tất cả' lại từ đầu.")

ERROR: Could not find a version that satisfies the requirement vnstock==4.0.5 (from versions: none)
ERROR: No matching distribution found for vnstock==4.0.5


ModuleNotFoundError: No module named 'vnstock'

## 🔧 Ô 2 — CÁC LỰA CHỌN CỦA BẠN (có thể sửa ở đây)
Bình thường **không cần sửa gì**. Nếu muốn đổi khung mẫu, chỉ sửa các dòng bên dưới.

In [ ]:
# === BẠN CÓ THỂ SỬA 3 DÒNG NÀY ===
SAN_GIU_LAI      = ['HSX', 'HOSE', 'HNX']      # Sàn muốn giữ (HSX = HOSE). Muốn thêm UPCoM: thêm 'UPCOM'
NAM_BAT_DAU      = 2014
NAM_KET_THUC     = 2023

# Từ khóa ngành TÀI CHÍNH cần LOẠI BỎ (giữ nguyên theo chuẩn nghiên cứu chi phí vốn)
TU_KHOA_LOAI_BO  = ['ngân hàng', 'bảo hiểm', 'chứng khoán', 'dịch vụ tài chính',
                    'môi giới', 'quỹ', 'bank', 'insurance', 'securities', 'financial service']
# Ghi chú: Bất động sản KHÔNG nằm trong danh sách loại bỏ => được GIỮ LẠI.
print('✅ Đã ghi nhận lựa chọn. Chuyển sang Ô 3.')

## 📥 Ô 3 — Lấy danh sách toàn bộ DN niêm yết + phân ngành

In [ ]:
# === Ô 3 (bản sửa): Lấy danh sách DN niêm yết + phân ngành ===
import pandas as pd
from vnstock import Listing

df = None
# Nguồn HỢP LỆ cho Listing chỉ gồm: 'kbs', 'vci', 'msn'. Ưu tiên 'kbs' (mặc định, ổn định nhất).
for src in ['kbs', 'vci', 'msn']:
    try:
        df = Listing(source=src).symbols_by_industries()
        if df is not None and len(df) > 0:
            print(f"✅ Lấy dữ liệu thành công từ nguồn: {src}  (số dòng: {len(df)})")
            break
    except Exception as e:
        print(f"⚠️  Nguồn '{src}' chưa được: {str(e)[:110]}")
        df = None

# Nếu vẫn thất bại thì DỪNG ngay tại đây (để Ô 4 không báo lỗi khó hiểu)
if df is None or len(df) == 0:
    raise RuntimeError(
        "❌ Vẫn chưa lấy được dữ liệu. Hãy COPY toàn bộ phần chữ ô này gửi Claude để đổi cách lấy."
    )

print("\n>>> KÍCH THƯỚC BẢNG:", df.shape)
print(">>> TÊN CÁC CỘT (nếu Ô 4 lỗi, gửi dòng này cho Claude):")
print(list(df.columns))
print("\n>>> XEM THỬ 3 DÒNG ĐẦU:")
display(df.head(3))

## 🧹 Ô 4 — Lọc mẫu (bỏ ngành tài chính, giữ HOSE+HNX)
Ô này tự dò tên cột nên chịu được khác biệt giữa các nguồn dữ liệu.

In [ ]:
cols_lower = {c.lower(): c for c in df.columns}

def tim_cot(tu_khoa_list):
    """Tìm tên cột chứa 1 trong các từ khóa."""
    for kw in tu_khoa_list:
        for low, orig in cols_lower.items():
            if kw in low:
                return orig
    return None

cot_ma      = tim_cot(['symbol', 'ticker', 'ma'])
cot_ten     = tim_cot(['organ_name', 'company', 'ten', 'name'])
cot_san     = tim_cot(['exchange', 'comgroupcode', 'board', 'san'])
# gom tất cả cột có thể chứa tên ngành (icb nhiều cấp)
cot_nganh   = [orig for low, orig in cols_lower.items()
               if any(k in low for k in ['icb', 'industry', 'nganh', 'sector'])]

print('Cột MÃ  :', cot_ma)
print('Cột TÊN :', cot_ten)
print('Cột SÀN :', cot_san)
print('Cột NGÀNH:', cot_nganh)

d = df.copy()

# 1) Lọc theo sàn (nếu có cột sàn)
if cot_san:
    san_norm = d[cot_san].astype(str).str.upper()
    giu = [s.upper() for s in SAN_GIU_LAI]
    d = d[san_norm.isin(giu)]
    print(f'\nSau lọc sàn {SAN_GIU_LAI}: còn {len(d)} DN')
else:
    print('\n⚠️  Không thấy cột sàn — bỏ qua bước lọc sàn (sẽ lọc lại sau).')

# 2) Loại ngành tài chính (dò trên mọi cột ngành)
if cot_nganh:
    text_nganh = d[cot_nganh].astype(str).agg(' '.join, axis=1).str.lower()
    mask_taichinh = text_nganh.apply(lambda t: any(kw in t for kw in TU_KHOA_LOAI_BO))
    print(f'Số DN ngành tài chính bị loại: {mask_taichinh.sum()}')
    d = d[~mask_taichinh]
else:
    print('⚠️  Không thấy cột ngành — chưa loại được tài chính. Gửi Claude tên cột ở Ô 3.')

print(f'\n✅ MẪU CUỐI CÙNG: {len(d)} doanh nghiệp')

# Bảng gọn để xuất
giu_cot = [c for c in [cot_ma, cot_ten, cot_san] + cot_nganh if c]
mau = d[giu_cot].drop_duplicates().reset_index(drop=True)
display(mau.head(10))

## 💾 Ô 5 — Xuất Excel + tạo danh sách mục tiêu crawl, lưu vào Google Drive

In [ ]:
from google.colab import drive, files
import os
drive.mount('/content/drive')

thu_muc = '/content/drive/MyDrive/CSRDI'
os.makedirs(thu_muc, exist_ok=True)

# Bảng 1: danh sách mẫu DN
duong_dan_mau = f'{thu_muc}/DanhSach_Mau_CSRDI.xlsx'
mau.to_excel(duong_dan_mau, index=False)

# Bảng 2: danh sách mục tiêu crawl = mỗi DN x mỗi năm 2014..2023
ma_list = mau[cot_ma].dropna().unique().tolist() if cot_ma else []
rows = [{'ma_ck': m, 'nam': y, 'trang_thai': 'chua_tai', 'ten_file': f'{m}_{y}.pdf'}
        for m in ma_list for y in range(NAM_BAT_DAU, NAM_KET_THUC + 1)]
muc_tieu = pd.DataFrame(rows)
duong_dan_muctieu = f'{thu_muc}/MucTieu_Crawl.xlsx'
muc_tieu.to_excel(duong_dan_muctieu, index=False)

print(f'✅ Đã lưu {len(mau)} DN vào: {duong_dan_mau}')
print(f'✅ Đã tạo {len(muc_tieu)} mục tiêu crawl (DN x năm) vào: {duong_dan_muctieu}')
print('\n📥 Đang tải file danh sách mẫu về máy bạn...')
files.download(duong_dan_mau)

---
## ✅ XONG NOTEBOOK #1
Bạn giờ có 2 file trong Google Drive (thư mục **CSRDI**):
- `DanhSach_Mau_CSRDI.xlsx` — danh sách DN đã lọc (mở ra xem thử, kiểm tra có hợp lý không)
- `MucTieu_Crawl.xlsx` — danh sách việc cần tải (DN × năm) cho bước sau

**Gửi lại cho Claude:** số lượng DN cuối cùng (dòng "MẪU CUỐI CÙNG: ... doanh nghiệp") để mình kiểm tra có gần 600 không, rồi mình dựng **Notebook #2 — Tải báo cáo**.

In [ ]:
# === Ô 6: Tinh chỉnh danh sách (bỏ nốt tài chính + lọc sàn HOSE/HNX nếu lấy được) ===
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

P = '/content/drive/MyDrive/CSRDI/'
mau = pd.read_excel(P + 'DanhSach_Mau_CSRDI.xlsx')
print("Trước tinh chỉnh:", len(mau), "DN")

# 1) Bỏ nốt các DN có tên ngành chứa 'tài chính' (gồm 'Tài chính khác': IPA, OGC, TVC)
truoc = len(mau)
mau = mau[~mau['industry_name'].astype(str).str.lower().str.contains('tài chính', na=False)]
print(f"→ Bỏ {truoc-len(mau)} DN nhóm tài chính còn sót. Còn:", len(mau))

# 2) Thử lấy thông tin SÀN để lọc HOSE+HNX (nếu lỗi thì bỏ qua, KHÔNG sao)
try:
    from vnstock import Listing
    ex = None
    for src in ['kbs', 'vci', 'msn']:
        try:
            ex = Listing(source=src).symbols_by_exchange()
            if ex is not None and len(ex) > 0:
                print(f"\nLấy bảng sàn từ '{src}'. Cột:", list(ex.columns)); break
        except Exception as e:
            print(f"  sàn nguồn {src} chưa được: {str(e)[:60]}"); ex = None
    if ex is not None and len(ex) > 0:
        low = {c.lower(): c for c in ex.columns}
        c_ma  = next((low[k] for k in low if any(t in k for t in ['symbol','ticker'])), None)
        c_san = next((low[k] for k in low if any(t in k for t in ['exchange','comgroup','board'])), None)
        print("  cột mã:", c_ma, "| cột sàn:", c_san)
        if c_ma and c_san:
            ex2 = ex[[c_ma, c_san]].copy(); ex2.columns = ['symbol','san']
            ex2['san'] = ex2['san'].astype(str).str.upper()
            print("  Phân bố sàn:", ex2['san'].value_counts().to_dict())
            mau = mau.merge(ex2, on='symbol', how='left')
            if mau['san'].notna().mean() > 0.7:
                giu = mau['san'].isin(['HOSE','HSX','HNX'])
                print(f"  → {giu.sum()} DN HOSE/HNX, loại {(~giu).sum()} DN sàn khác")
                mau = mau[giu]
            else:
                print("  ⚠️ Ghép sàn không đủ tin cậy → giữ nguyên, lọc sàn ở bước sau")
            mau = mau.drop(columns=['san'], errors='ignore')
except Exception as e:
    print("  (Bỏ qua lọc sàn:", str(e)[:80], ")")

mau = mau.drop_duplicates('symbol').reset_index(drop=True)
print("\n✅ SỐ DN CUỐI CÙNG:", len(mau))

# 3) Ghi đè lại 2 file
mau.to_excel(P + 'DanhSach_Mau_CSRDI.xlsx', index=False)
rows = [{'ma_ck': m, 'nam': y, 'trang_thai': 'chua_tai', 'ten_file': f'{m}_{y}.pdf'}
        for m in mau['symbol'] for y in range(2014, 2024)]
pd.DataFrame(rows).to_excel(P + 'MucTieu_Crawl.xlsx', index=False)
print("✅ Đã cập nhật lại 2 file trong Drive.")

In [ ]:
# === Ô 7: Thêm cột SÀN vào file danh sách (để lưu lại, dễ tra cứu) ===
import pandas as pd
from vnstock import Listing
P = '/content/drive/MyDrive/CSRDI/'

mau = pd.read_excel(P + 'DanhSach_Mau_CSRDI.xlsx')          # 627 DN đã lọc
ex  = Listing(source='kbs').symbols_by_exchange()[['symbol', 'exchange']]
ex.columns = ['symbol', 'san']

mau = mau.merge(ex, on='symbol', how='left')
# đưa cột 'san' lên ngay sau mã cho dễ nhìn
cols = ['symbol', 'san'] + [c for c in mau.columns if c not in ['symbol', 'san']]
mau = mau[cols]

mau.to_excel(P + 'DanhSach_Mau_CSRDI.xlsx', index=False)
print("✅ Đã thêm cột 'san'. Phân bố sàn của mẫu:", mau['san'].value_counts().to_dict())
print("Tổng:", len(mau), "DN")
print(mau.head(8).to_string())

In [ ]:
# === Ô 7: Thêm cột SÀN vào CẢ 2 file (danh sách mẫu + mục tiêu crawl) ===
import pandas as pd
from vnstock import Listing
P = '/content/drive/MyDrive/CSRDI/'

# Lấy bảng ánh xạ mã -> sàn (1 lần)
ex = Listing(source='kbs').symbols_by_exchange()[['symbol', 'exchange']]
ex.columns = ['symbol', 'san']

# ----- File 1: DanhSach_Mau -----
mau = pd.read_excel(P + 'DanhSach_Mau_CSRDI.xlsx')
mau = mau.drop(columns=['san'], errors='ignore').merge(ex, on='symbol', how='left')
mau = mau[['symbol', 'san'] + [c for c in mau.columns if c not in ['symbol', 'san']]]
mau.to_excel(P + 'DanhSach_Mau_CSRDI.xlsx', index=False)

# ----- File 2: MucTieu_Crawl (khóa ghép là ma_ck) -----
muc = pd.read_excel(P + 'MucTieu_Crawl.xlsx')
ex2 = ex.rename(columns={'symbol': 'ma_ck'})
muc = muc.drop(columns=['san'], errors='ignore').merge(ex2, on='ma_ck', how='left')
muc = muc[['ma_ck', 'san', 'nam', 'trang_thai', 'ten_file']]
muc.to_excel(P + 'MucTieu_Crawl.xlsx', index=False)

print("✅ Đã thêm cột 'san' vào cả 2 file.")
print("Phân bố sàn (theo DN):", mau['san'].value_counts().to_dict())
print("Số DN:", len(mau), "| Số dòng mục tiêu crawl:", len(muc))
print("\nMẫu file mục tiêu:")
print(muc.head(6).to_string())